In [1]:
import pandas as pd
import numpy
import json
import os
from dotenv import load_dotenv
from copy import deepcopy
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))
from src.utils.eval_utils import metrics_instructabsa, parse_absa_string, parse_aoste, calculate_metrics

/home/ext_hanif_zhafran07_gmail_com/testing/absa-sft-comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import List, Dict
def calculate_metrics(predictions: List[List[Dict[str, str]]], targets: List[List[Dict[str, str]]], task='') -> Dict[str, float]:
	"""
	Calculate precision, recall, and F1 score for the given predictions and targets for ABSA.

	Args:
		predictions (List[List[Dict[str, str]]]): List of predicted triplets.
		targets (List[List[Dict[str, str]]]): List of target triplets.
		task (str): The task name for which metrics are calculated.
	
	Returns:
		Dict[str, float]: A dictionary containing precision, recall, and F1 score.
	"""
	true_positive = 0
	false_positive = 0
	false_negative = 0
	for prediction,target in zip(predictions,targets):
		for target_tuple in target:
			if target_tuple in prediction:
				true_positive += 1
			else:
				false_negative += 1
		false_positive += sum(1 for pred in prediction if pred not in target)
	precision = true_positive/(true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
	recall = true_positive/(true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
	f1 = (2 * recall * precision)/(recall + precision) if (recall + precision) > 0 else 0
	return {
		f"precision_{task}" : precision,
		f"recall_{task}" : recall,
		f"f1_{task}" : f1
	}

def form_aoste_triplet(triplets: List[List[Dict[str, str]]]) -> str:
	"""
	Form AOSTE triplets from the given predictions.

	Args:
		predictions (List[List[Dict[str, str]]]): List of predicted triplets.
	Returns:
		str: A string representation of the AOSTE triplets, the format is {aspect}:{opinion}:{sentiment},{aspect}:{opinion}:{sentiment}.
	"""
	aoste_triplets = []
	for triplet in triplets:
		aspect = triplet.get('A', '')
		opinion = triplet.get('O', '')
		sentiment = triplet.get('S', '')
		aoste_triplets.append(f"{aspect}:{opinion}:{sentiment}")
	return ', '.join(aoste_triplets)

In [3]:
form_aoste_triplet([{'A': 'over all', 'O': 'baik', 'S': 'positive'}, {'A': 'air hot waternya', 'O': 'akan lebih memuaskan jika air hot waternya bisa nyala 24jam', 'S': 'negative'}])

'over all:baik:positive, air hot waternya:akan lebih memuaskan jika air hot waternya bisa nyala 24jam:negative'

### InstructABSA to Exact Match

In [4]:
raw_instructabsa_path = 'dataset/aoste/googlemt5-base-hoasa_hotel-aoste-20260115_101411/raw_inference_results.csv'
raw_data = pd.read_csv(raw_instructabsa_path)
raw_data.head()

,text,labels,pred_labels
0,Definition: You will be given an Indonesian se...,pelayanan nya:sangat ramah:positive,pelayanan nya:sangat ramah:positive
1,Definition: You will be given an Indonesian se...,wifi:tidak bagus harus keluar kamar:negative,wifi:tidak bagus harus keluar kamar:negative
2,Definition: You will be given an Indonesian se...,kamarnya:beda:negative,kamarnya:beda:negative
3,Definition: You will be given an Indonesian se...,"over all:baik:positive, air hot waternya:akan ...","over all:baik:positive, air hot waternya:bisa ..."
4,Definition: You will be given an Indonesian se...,fasilatas:sesuia:positive,fasilitatas:suia:positive


In [5]:
raw_data['parsed_labels'] = raw_data['labels'].apply(lambda x: x.split(','))
raw_data['parsed_pred_labels'] = raw_data['pred_labels'].apply(lambda x: x.split(','))

In [6]:
results = calculate_metrics(raw_data['parsed_pred_labels'].to_list(), raw_data['parsed_labels'].to_list(), task='instructabsa')
for metric_name, metric_value in results.items():
	metric_value *= 100
	print(f"{metric_name}: {metric_value:.2f}%")

precision_instructabsa: 65.16%
recall_instructabsa: 63.79%
f1_instructabsa: 64.47%


In [7]:
metrics_instructabsa(raw_data['labels'], raw_data['pred_labels'])

(0.7145688800792864, 0.7, 0.7072094163805788, None)

In [8]:
raw_data['labels']

0                     pelayanan nya:sangat ramah:positive
1            wifi:tidak bagus harus keluar kamar:negative
2                                  kamarnya:beda:negative
3       over all:baik:positive, air hot waternya:akan ...
4                               fasilatas:sesuia:positive
                              ...                        
1281                            ac nya:gk dingin:negative
1282    kamarnya:enak luas:positive, sprei:masih lemba...
1283    kebersihan:sangat tidak baik:negative, kamar:b...
1284    pelayanan staf hotel:baik:positive, layanan ho...
1285    harga:cukup murah:positive, pelayanan:baik:pos...
Name: labels, Length: 1286, dtype: object

### Exact Match to InstructABSA

In [11]:
from glob import glob
paths = glob('outputs/evals/hotel_reviews/*/mvp/seed_*/*/*/*/*/voting_results.json')
paths += glob('outputs/evals/hotel_reviews/*/mvp_aos/seed_*/*/*/*/*/inference_results.json')
paths

['outputs/evals/hotel_reviews/jav/mvp/seed_123/20260402_134023_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/voting_results.json',
 'outputs/evals/hotel_reviews/jav/mvp/seed_31415/20260402_152635_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/voting_results.json',
 'outputs/evals/hotel_reviews/jav/mvp/seed_9584/20260402_124702_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/voting_results.json',
 'outputs/evals/hotel_reviews/jav/mvp/seed_777/20260402_161858_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/voting_results.json',
 'outputs/evals/hotel_reviews/jav/mvp/seed_2024/20260402_143335_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/voting_results.json',
 'outputs/evals/hotel_reviews/eng/mvp/seed_123/2026

In [12]:
data_metrics = []
for output_path in paths:
	lang = output_path.split('/')[3]
	seed = output_path.split('/')[5].split('_')[1]
	dataset_folder = output_path.split('/')[4]
	use_constrained_decoding = output_path.split('/')[-2]
	if use_constrained_decoding == 'constrained_decoding':
		use_constrained_decoding = True
	else:
		use_constrained_decoding = False
	with open(output_path, 'r') as f:
		data = json.load(f)
	df_data_output = pd.DataFrame(data)
	df_data_output['target_aoste'] = df_data_output['target'].apply(lambda x: form_aoste_triplet(parse_absa_string(x)))
	df_data_output['prediction_aoste'] = df_data_output['prediction'].apply(lambda x: form_aoste_triplet(parse_absa_string(x)))
	prec, rec, f1, _ = metrics_instructabsa(df_data_output['target_aoste'].to_list(), df_data_output['prediction_aoste'].to_list())
	data_metrics.append({
		'lang': lang,
		'seed': seed,
		'dataset_folder': dataset_folder,
		'use_constrained_decoding': use_constrained_decoding,
		'precision': prec,
		'recall': rec,
		'f1': f1
	})

In [13]:
df_metrics_aoste = pd.DataFrame(data_metrics)
df_metrics_aoste.loc[df_metrics_aoste['use_constrained_decoding'] == False]

,lang,seed,dataset_folder,use_constrained_decoding,precision,recall,f1
0,jav,123,mvp,False,0.614555,0.590100,0.602080
1,jav,31415,mvp,False,0.611392,0.586865,0.598878
2,jav,9584,mvp,False,0.613045,0.586865,0.599669
3,jav,777,mvp,False,0.621968,0.597218,0.609341
4,jav,2024,mvp,False,0.611545,0.582659,0.596753
5,eng,123,mvp,False,0.707161,0.692982,0.700000
6,eng,31415,mvp,False,0.706863,0.690476,0.698574
7,eng,9584,mvp,False,0.703928,0.690476,0.697137
8,eng,777,mvp,False,0.714102,0.696429,0.705155
9,eng,2024,mvp,False,0.716104,0.704887,0.710452


In [ ]:
# Store the metrics in a csv file